In [ ]:
# Week 4 — Logistic Regression + Feature Scaling with full evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# -------------------- Load & target engineering --------------------
df = pd.read_csv("diabetes_012.csv")
TARGET = "Diabetes_012"

# Binary target: 0 stays 0; {1,2} -> 1 (pre/diabetes)
y = (df[TARGET] > 0).astype(int)
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = X.columns.tolist()
std_scaler = ColumnTransformer([("num", StandardScaler(), num_cols)], remainder="drop")
mm_scaler  = ColumnTransformer([("num", MinMaxScaler(),  num_cols)], remainder="drop")

#  Pipelines 
pipe_none = Pipeline([("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))])
pipe_std  = Pipeline([("scale", std_scaler),
                      ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))])
pipe_mm   = Pipeline([("scale", mm_scaler),
                      ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))])

#  Cross-validation comparison 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def cv_report(name, model, scoring="roc_auc"):
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"{name:>18} | {scoring}: {scores.mean():.3f} ± {scores.std():.3f}")

print("Cross-validated comparison (higher is better):")
for nm, mdl in [("No scaling", pipe_none), ("StandardScaler", pipe_std), ("MinMaxScaler", pipe_mm)]:
    cv_report(nm, mdl, scoring="roc_auc")

#  Hyperparameter tuning on StdScaler pipeline 
param_grid = {"clf__penalty": ["l2"], "clf__C": [0.1, 0.5, 1, 2, 5, 10]}
gs = GridSearchCV(pipe_std, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
gs.fit(X_train, y_train)
best_model = gs.best_estimator_
print("\nBest params:", gs.best_params_)

#  Evaluation 
def evaluate(model, X_te, y_te, name="Model"):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    auc = roc_auc_score(y_te, y_proba)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}")
    print(f"F1       : {f1:.3f}")
    print(f"ROC-AUC  : {auc:.3f}")
    print("\nConfusion Matrix:\n", confusion_matrix(y_te, y_pred))
    print("\nClassification Report:\n", classification_report(y_te, y_pred, digits=3, zero_division=0))

    fpr, tpr, _ = roc_curve(y_te, y_proba)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {name}")
    plt.legend()
    plt.show()

evaluate(best_model, X_test, y_test, name="Tuned Logistic (StdScaler)")
# Coefficients correspond to standardized features
clf = best_model.named_steps["clf"]
coef = pd.Series(clf.coef_[0], index=num_cols)
coef_abs_sorted = coef.reindex(coef.abs().sort_values(ascending=False).index)

print("\nTop 20 features by |coefficient| (StdScaler):")
print(coef_abs_sorted.head(20).to_frame("Coefficient"))
print("\nTop 10 positive drivers (increase odds of class=1):")
print(coef.sort_values(ascending=False).head(10).to_frame("Coefficient"))
print("\nTop 10 negative drivers (decrease odds of class=1):")
print(coef.sort_values().head(10).to_frame("Coefficient"))


Cross-validated comparison (higher is better):
        No scaling | roc_auc: 0.818 ± 0.001
    StandardScaler | roc_auc: 0.818 ± 0.001
      MinMaxScaler | roc_auc: 0.818 ± 0.001
